# 7주차 ② CNN 구조의 발전과 CIFAR-10 구현 — 실습 5~7  〔교수용 정답본〕

> ⚠️ **이 파일은 교수용입니다.** 학생에게는 `12_cnn_cifar10_blank.ipynb` 를 배포하세요.
> 빈칸 9곳 · 20줄이 모두 채워져 있습니다.

---

## 들어가기 전에 — 사람들이 찾아낸 세 가지 답

1교시에서 재료를 다 얻었습니다: `Conv2d`, `MaxPool2d`, 그리고 출력 크기 공식.
남은 질문은 하나입니다 — **어떤 순서로, 몇 번 쌓을 것인가?**

| 구조 | 연도 | 핵심 아이디어 | 무엇을 해결했나 |
|---|---|---|---|
| **LeNet-5** | 1998 | `Conv → Pool` 을 번갈아 → FC | CNN 의 기본형을 확립 |
| **VGG** | 2014 | **3×3 커널만** 깊게 (16~19층) | 큰 커널 하나보다 작은 것 여러 개가 낫다 |
| **ResNet** | 2015 | **잔차 연결(skip connection)** | **깊으면 오히려 학습이 안 되는 문제** ★ |

```
   [VGG 의 발견]  5×5 커널 1개  vs  3×3 커널 2개

      보는 범위는 같다.   파라미터는 25개 vs 18개 (적다)
      게다가 활성함수가 2번 들어간다 → 표현력이 더 좋다
      → "3×3 만 쓰고 깊게 쌓자"
```

### 잔차 연결 — 깊이의 벽 ★

```
   깊게 쌓았더니 오히려 성능이 떨어졌다 (56층 < 20층)

      원인: 층이 깊어 기울기가 앞까지 못 온다 (5주차의 기울기 소실과 같은 뿌리)

   [해법 — 지름길을 놓는다]

        x ──────────────────────┐
        │                       │ (그대로 더한다)
        └→ Conv → BN → ReLU → Conv → BN →(+)→ ReLU
                                       ↑
                          출력 = F(x) + x
```

> **핵심 ★ (출제 지점)**: 잔차 연결이 해결한 문제는 **"깊은 망의 학습 실패"** 입니다.
> `+x` 덕분에 **기울기가 지름길로 앞까지 곧장 흐릅니다.**
> 그리고 층이 최악의 경우 `F(x)=0` 을 배우면 **그냥 입력을 통과**시키므로,
> **깊게 쌓아도 손해는 안 봅니다.**

> **포인트**: 이 `+x` 한 줄이 **11주차 트랜스포머에도 그대로** 들어갑니다
> (`잔차 연결 + LayerNorm`). 현대 딥러닝의 거의 모든 구조가 이걸 씁니다.

> **포인트**: 5주차의 `nn.Sequential` 로는 이 구조를 못 씁니다 — 건너뛰는 연결이 있으니까요.
> **`nn.Module` 을 상속해 `forward` 를 직접 쓰는 이유**가 여기서 확실해집니다.
> 오늘 우리가 만들 CNN 도 `nn.Module` 로 씁니다.

> 오늘 ResNet 을 만들지는 않습니다. **9주차 전이학습에서 사전학습된 ResNet 을 그대로 가져다 씁니다.**

---

## 실습 5 — CIFAR-10 준비와 CNN 작성 ★★ (17분)

### CIFAR-10 이란

| | FashionMNIST (5·6주차) | **CIFAR-10 (오늘)** |
|---|---|---|
| 크기 | 28×28 | **32×32** |
| 채널 | 1 (흑백) | **3 (컬러)** |
| 클래스 | 10 | 10 |
| 훈련/테스트 | 60,000 / 10,000 | 50,000 / 10,000 |
| 난이도 | 쉬움 (MLP 로 86%) | **어려움** — 배경·각도·색이 제각각 |

**채널이 1 → 3 으로 바뀌는 것**이 첫 번째 함정입니다. `Conv2d` 의 `in_channels` 를 3 으로.

In [ ]:
# 셀 1 — 데이터 준비
import torch, torch.nn as nn, time, os
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("장치 :", device)

# 실습실 GPU 면 원래 크기로, CPU 면 줄여서 돈다  ★ 숫자를 바꾸지 마세요
if device.type == "cuda":
    N_TRAIN, N_VAL, EPOCHS = 45000, 5000, 15
else:
    N_TRAIN, N_VAL, EPOCHS = 10000, 2000, 8
print(f"훈련 {N_TRAIN}장 / 검증 {N_VAL}장 / {EPOCHS} epoch")

CLASSES = ["비행기","자동차","새","고양이","사슴","개","개구리","말","배","트럭"]

tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

full = datasets.CIFAR10("data", train=True,  download=True, transform=tf)
test = datasets.CIFAR10("data", train=False, download=True, transform=tf)

train_set, val_set, _ = random_split(full, [N_TRAIN, N_VAL, len(full) - N_TRAIN - N_VAL],
                                     generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256)
test_loader  = DataLoader(test,      batch_size=256)

xb, yb = next(iter(train_loader))
print("배치 shape :", tuple(xb.shape), "| 정답 shape :", tuple(yb.shape))
print(f"정규화 확인  mean {xb.mean():.4f} / std {xb.std():.4f}")
print(f"배치 수      훈련 {len(train_loader)} / 검증 {len(val_loader)}")

> **관찰 포인트 ★**: FashionMNIST 는 `(B,1,28,28)` 이었습니다. CIFAR-10 은 **`(B,3,32,32)`** —
> **채널이 3개(컬러)** 이고 크기도 큽니다. 그래서 `Conv2d` 의 `in_channels=3` 으로 시작합니다.

> **포인트 ★**: `Normalize` 의 값이 **3개씩**입니다 — R, G, B 채널마다 따로.
> 5주차 FashionMNIST 는 흑백이라 `(0.2860,), (0.3530,)` 처럼 **1개씩**이었습니다.

> **포인트**: 6주차에 배운 **훈련/검증/테스트 세 분할**을 처음부터 적용했습니다.
> 증강 효과(3교시)를 **검증 곡선으로 판정**해야 하기 때문입니다.
> **테스트 세트를 보면서 모델을 고르면 그건 이미 테스트가 아닙니다.**

이제 모델입니다. 블록을 **세 번** 쌓습니다.

In [ ]:
# 셀 2 — CNN 정의  ★ 오늘의 핵심
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # [블록 1]  32×32 → 16×16,  채널 3 → 32
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # ★ 크기 유지
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # [블록 2]  16×16 → 8×8,  채널 32 → 64
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # [블록 3]  8×8 → 4×4,  채널 64 → 128
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),                              # 6주차의 그것
            nn.Linear(128 * 4 * 4, 256),                  # 128×4×4 = 2048
            nn.ReLU(),
            nn.Linear(256, num_classes),                  # ★ Softmax 를 붙이지 않는다
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x)


model = CNN().to(device)
print(model)

In [ ]:
# 셀 3 — 가짜 입력으로 shape 검산  ★ 학습 전에 반드시
dummy = torch.randn(2, 3, 32, 32).to(device)

x = dummy
for name in ["block1", "block2", "block3"]:
    x = getattr(model, name)(x)
    print(f"  {name} 후 : {tuple(x.shape)}")
print("최종 출력 :", tuple(model(dummy).shape), "  ← (2, 10) 이어야 한다")

def 세기(m): return sum(p.numel() for p in m.parameters())
print()
print(f"파라미터  블록1 {세기(model.block1):>8,}")
print(f"          블록2 {세기(model.block2):>8,}")
print(f"          블록3 {세기(model.block3):>8,}")
print(f"          FC    {세기(model.classifier):>8,}   ← 여기가 대부분이다")
print(f"          합계  {세기(model):>8,}")

> **핵심 ★★ — 마지막 빈칸이 이 실습의 고비입니다.**
> ```
>   입력 32×32 → 블록1 (pool) → 16×16 → 블록2 (pool) → 8×8 → 블록3 (pool) → 4×4
>   마지막 채널 128,  공간 4×4
>   →  Flatten 하면  128 × 4 × 4 = 2048
> ```
> **1교시 실습 2 의 출력 크기 공식**이 여기서 쓰입니다.
> `padding=1, kernel=3` 은 크기를 유지하므로, **줄이는 것은 오직 `MaxPool2d(2)` 세 번**입니다.
> `32 → 16 → 8 → 4`.

> **관찰 포인트 ★**: 블록별 출력이 `(2,32,16,16) → (2,64,8,8) → (2,128,4,4)` 로 나옵니다.
> **빈칸에 넣은 2048 이 맞았는지** 여기서 바로 확인됩니다.

> **막히면**:
> | 증상 | 원인 |
> |---|---|
> | `mat1 and mat2 shapes cannot be multiplied (2x2048 and 4096x256)` | 빈칸의 숫자가 틀렸다. 오류 메시지의 **앞 숫자(2048)** 가 정답 |
> | `Expected 3 channels, got 1` | `in_channels` 를 1 로 뒀다 (FashionMNIST 습관) |
> | `BatchNorm2d` 오류 | `BatchNorm1d` 를 썼다. 이미지는 **2d** |
>
> **오류 메시지가 정답을 알려 줍니다.** PyTorch 오류는 끝까지 읽으라고 3주차에 했던 그 이야기입니다.

> **관찰 포인트 ★★ — 파라미터가 어디에 몰려 있나**
>
> 1교시에서 *"MLP 첫 층만 157만 개"* 라고 했죠. **CNN 은 전체로도 그보다 적습니다.**
>
> 그런데 출력을 잘 보세요 — **합성곱 세 블록을 다 합친 것보다 마지막 FC 가 훨씬 큽니다.**
> *"파라미터를 잡아먹는 건 합성곱이 아니라 FC"* — 실습 7의 복선입니다.
>
> 그래서 ResNet 이후 구조는 **FC 를 거의 없애고** `AdaptiveAvgPool2d(1)` 로
> 특징맵을 채널당 숫자 하나로 눌러 버립니다. **9주차 이후 모델들이 다 그렇습니다.**

> **왜 BatchNorm 이 들어갔나** (6주차): 층마다 분포를 다시 맞춰 줘서
> **학습이 빨라지고 학습률을 크게 써도 안정적**입니다. `Dropout(0.3)` 은 과적합 방지.

> **체크포인트**: `model(dummy).shape` 이 `(2, 10)` 이면 통과.

---

## 실습 6 — 학습 실행 + 곡선 확인 (12분)

**루프 자체는 5·6주차와 똑같습니다.** 달라진 건 모델뿐입니다.
```
   ① 순전파  → ② 손실 → ③ 기울기 지우기 → ④ 역전파 → ⑤ 갱신
```
옵티마이저와 스케줄러는 **6주차에서 고른 조합**(AdamW + Cosine)을 그대로 씁니다.

In [ ]:
# 셀 4 — 학습 루프 (6주차 루프 그대로)
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


@torch.no_grad()
def evaluate(m, loader):
    m.eval()
    run = 0.0; c = t = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = m(xb)
        run += loss_fn(out, yb).item() * yb.size(0)
        pred = out.argmax(dim=1)
        c += (pred == yb).sum().item(); t += yb.size(0)
    return run / t, c / t


tr_hist, va_hist, va_acc, lr_hist = [], [], [], []
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train(); run = n = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        out  = model(xb)
        loss = loss_fn(out, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        run += loss.item() * yb.size(0); n += yb.size(0)

    scheduler.step()                       # ★ epoch 끝에서 한 번 (배치마다 아님)
    vl, va = evaluate(model, val_loader)
    tr_hist.append(run/n); va_hist.append(vl); va_acc.append(va)
    lr_hist.append(scheduler.get_last_lr()[0])
    print(f"epoch {epoch:2d}/{EPOCHS} | 훈련 {run/n:.4f} | 검증 {vl:.4f} "
          f"| 검증정확도 {va*100:5.2f}% | lr {lr_hist[-1]:.5f} | {time.time()-t0:5.1f}s")

print()
print(f"학습 시간 : {time.time()-t0:.1f}초")
print(f"테스트 정확도 : {evaluate(model, test_loader)[1]*100:.2f}%   ← 마지막에 딱 한 번")

In [ ]:
# 셀 5 — 곡선 + 저장
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(range(1, EPOCHS+1), tr_hist, "o-", label="훈련")
ax[0].plot(range(1, EPOCHS+1), va_hist, "o-", label="검증")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
ax[0].set_title("손실"); ax[0].grid(alpha=.3)
ax[1].plot(range(1, EPOCHS+1), [a*100 for a in va_acc], "o-", color="tab:green")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("%")
ax[1].set_title("검증 정확도"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/cnn_cifar10.pt")
print("저장 완료 : models/cnn_cifar10.pt")

> **관찰 포인트 ★**: 6주차에 배운 대로 **두 곡선을 함께** 봅니다.
> 훈련 손실은 계속 내려가는데 **검증 손실이 어느 순간부터 안 내려가거나 올라가면 과적합**입니다.
> **지금 그 갈라짐을 눈에 담아 두세요.** 3교시 증강 비교의 기준선입니다.

> **막히면**: 손실이 2.3 에서 안 내려가면 — ① 5단계 순서 확인
> ② `optimizer.zero_grad()` 가 `loss.backward()` **앞**에 있는지
> ③ `xb, yb = xb.to(device), yb.to(device)` 를 빠뜨리지 않았는지.

> **체크포인트**: 곡선 두 장이 그려지고 `models/cnn_cifar10.pt` 가 저장되면 통과.

---

## 실습 7 — MLP 와 비교 ★ (7분)

**오늘의 질문에 답할 차례입니다.** 5·6주차의 MLP 구조를 그대로 가져와
**같은 데이터·같은 epoch** 로 돌립니다. 파라미터 수도 함께 봅니다.

In [ ]:
# 셀 6 — 같은 데이터, 같은 조건으로 MLP
torch.manual_seed(42)

mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3*32*32, 512), nn.ReLU(),
    nn.Linear(512, 256), nn.ReLU(),
    nn.Linear(256, 10),
).to(device)

opt2 = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    mlp.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(mlp(xb), yb)
        opt2.zero_grad(); loss.backward(); opt2.step()
    print(f"epoch {epoch:2d}/{EPOCHS} | MLP 검증 정확도 {evaluate(mlp, val_loader)[1]*100:5.2f}%")
print(f"MLP 학습 시간 : {time.time()-t0:.1f}초")

mlp_acc = evaluate(mlp, test_loader)[1]
cnn_acc = evaluate(model, test_loader)[1]

print()
print(f"{'':6s} {'파라미터':>12s} {'테스트 정확도':>14s}")
print(f"{'MLP':6s} {sum(p.numel() for p in mlp.parameters()):>12,d} {mlp_acc*100:>13.2f}%")
print(f"{'CNN':6s} {sum(p.numel() for p in model.parameters()):>12,d} {cnn_acc*100:>13.2f}%")

> **관찰 포인트 ★★**: **파라미터는 CNN 이 더 적은데 정확도는 더 높습니다.**
>
> 1교시에서 말한 두 가지가 그대로 확인됩니다:
> 1. **공간 정보 소실** — 펴는 순간 옆 픽셀과의 관계가 사라진다
> 2. **가중치 공유** — CNN 은 같은 커널을 이미지 전체에 쓰므로 적은 파라미터로 더 잘 본다
>
> **크기가 힘이 아니라 구조가 힘입니다.**

> **핵심 ★**: *"같은 정확도라면 파라미터가 적은 모델이 유리하다"* — 메모리·속도·과적합 모두에서요.
> 그런데 여기서는 **파라미터가 적으면서 정확도도 높습니다.**
> **구조가 문제에 맞으면 그런 일이 일어납니다.** 9주차 전이학습도 같은 이야기입니다.

> **참고**: FashionMNIST 는 MLP 로도 86% 가 나왔습니다(5주차).
> 흑백이고 물체가 가운데 정렬돼 있어서 **펴도 손해가 적었기 때문**입니다.
> CIFAR-10 처럼 **배경과 위치가 제각각인 컬러 사진**에서 차이가 벌어집니다.

> **체크포인트**: 비교표가 출력되고, CNN 이 **더 적은 파라미터로 더 높은 정확도**를 낸 것을
> 확인하면 통과. **과제 제출물입니다.**

---

## 2교시 핵심 정리

| 개념 | 한 줄 정의 |
|------|-----------|
| **LeNet** | `Conv → Pool` 번갈아 + FC. 기본형 |
| **VGG** | **3×3 만 깊게**. 큰 커널 1개보다 작은 것 여러 개 |
| **ResNet / 잔차 연결** ★ | `출력 = F(x) + x`. **깊은 망의 학습 실패**를 해결 |
| **기본 블록** | `Conv → BN → ReLU → Pool` |
| **FC 입력 크기** ★★ | 마지막 채널 × H × W. **풀링 횟수로 H·W 를 센다** |
| **파라미터가 몰리는 곳** | 합성곱이 아니라 **마지막 FC** |
| **CNN vs MLP** ★★ | 파라미터는 **적고** 정확도는 **높다** |

### 반드시 외우기 ★

```python
nn.Sequential(
    nn.Conv2d(in_ch, out_ch, 3, padding=1),
    nn.BatchNorm2d(out_ch), nn.ReLU(),
    nn.MaxPool2d(2),
)
# 32 → 16 → 8 → 4  (풀링 3번)   →  Flatten 하면 마지막채널 × 4 × 4
```

### 2교시 체크리스트

- [ ] VGG 가 3×3 만 쓰는 이유를 안다
- [ ] **잔차 연결이 해결한 문제**를 말할 수 있다 ★
- [ ] `Conv-BN-ReLU-Pool` 블록을 직접 썼다
- [ ] **FC 입력 크기 2048 을 풀링 횟수로 계산**했다 ★★
- [ ] 학습 전에 가짜 입력으로 shape 을 검산했다
- [ ] CIFAR-10 CNN 을 학습시켜 곡선을 봤다
- [ ] **MLP 와 파라미터 수·정확도를 비교**했다 ★
- [ ] `models/cnn_cifar10.pt` 를 저장했다

### 저장

`12_cnn_cifar10.ipynb` 로 저장합니다. **과제 제출물입니다.**

> **다음 시간 예고**: 검증 곡선이 후반에 갈라지는 것을 봤죠 — 과적합입니다.
> 3교시에는 두 가지로 더 짜냅니다 — **데이터를 늘리고(증강)**, **계산을 가볍게(AMP fp16)**.
> AMP 는 **8GB GPU 실습의 핵심 기법**이라 9주차·14주차에서 계속 씁니다.
> 그리고 **1~7주 정리 자료와 중간고사 안내**가 있습니다. 끝까지 남아 주세요.